# Reward-Weight Model Comparison

This notebook compares the saved PPO models trained with different runtime weights. It evaluates each model on the same validation/test splits and compares them against simple baselines so you can choose the reward weight with the best runtime/reduction tradeoff.


In [ ]:
from pathlib import Path
import os
import re
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from stable_baselines3 import PPO

RANDOM_STATE = 42
DEFAULT_RUNTIME_WEIGHT = 0.20

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "trajectories_db_analysis" / "one_and_two_opt_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

CHAIN_OPTIMIZER_DIR = PROJECT_ROOT / "machine_learning" / "RL" / "chain_optimizer"
sys.path.insert(0, str(CHAIN_OPTIMIZER_DIR))
from optimizer_chain_env import METRIC_NAMES, OptimizerChainEnv

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


## Load Data and Split

This recreates the same 80/10/10 train/validation/test split used by the training notebook.

In [ ]:
CSV_PATH = PROJECT_ROOT / "trajectories_db_analysis" / "one_and_two_opt_master.csv"
df = pd.read_csv(CSV_PATH)

df["two_qubit_gate_reduction"] = (
    df["initial_two_qubit_gates"] - df["final_two_qubit_gates"]
)
df["two_qubit_gate_reduction_pct"] = (
    100
    * df["two_qubit_gate_reduction"]
    / df["initial_two_qubit_gates"].replace(0, np.nan)
)

df = df.dropna(
    subset=[
        "two_qubit_gate_reduction_pct",
        "total_runtime_seconds",
        "final_two_qubit_gates",
    ]
).copy()

OPTIMIZER_NAMES = [
    "qiskit_standard",
    "qiskit_ai",
    "tket",
    "wisq_rules",
    "wisq_bqskit",
]
CATEGORY_NAMES = sorted(df["original_category"].unique())

circuits = np.array(sorted(df["original_circuit"].unique()))
train_val_circuits, test_circuits = train_test_split(
    circuits,
    test_size=0.10,
    random_state=RANDOM_STATE,
)
train_circuits, val_circuits = train_test_split(
    train_val_circuits,
    test_size=0.10 / 0.90,
    random_state=RANDOM_STATE,
)

train_df = df[df["original_circuit"].isin(train_circuits)].copy()
val_df = df[df["original_circuit"].isin(val_circuits)].copy()
test_df = df[df["original_circuit"].isin(test_circuits)].copy()

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "circuits": [len(train_circuits), len(val_circuits), len(test_circuits)],
        "rows": [len(train_df), len(val_df), len(test_df)],
    }
)
split_summary


## Reward Columns and Observation Scaling

Each model was trained with a different runtime weight. For fair comparison, we keep the raw metrics fixed and add utility columns for each weight.

In [ ]:
train_runtime_99 = float(train_df["total_runtime_seconds"].quantile(0.99))
runtime_denominator = np.log1p(max(train_runtime_99, 1e-9))


def add_reward_columns(frame, runtime_weight=DEFAULT_RUNTIME_WEIGHT):
    frame = frame.copy()
    frame["pct_score"] = frame["two_qubit_gate_reduction_pct"] / 100.0
    frame["runtime_score"] = (
        np.log1p(frame["total_runtime_seconds"].clip(lower=0))
        / runtime_denominator
    ).clip(0, 2)
    frame["utility_reward"] = frame["pct_score"] - runtime_weight * frame["runtime_score"]
    return frame


def utility_for_weight(frame, runtime_weight):
    return frame["pct_score"] - runtime_weight * frame["runtime_score"]


train_df = add_reward_columns(train_df, DEFAULT_RUNTIME_WEIGHT)
val_df = add_reward_columns(val_df, DEFAULT_RUNTIME_WEIGHT)
test_df = add_reward_columns(test_df, DEFAULT_RUNTIME_WEIGHT)


def initial_metric_frame(frame):
    return pd.DataFrame(
        {
            "num_qubits": frame["initial_num_qubits"],
            "depth": frame["initial_depth"],
            "two_qubit_gates": frame["initial_two_qubit_gates"],
            "total_gates": frame["initial_total_gates"],
            "two_qubit_depth": frame["initial_two_qubit_depth"],
            "gate_density": frame["initial_gate_density"],
            "two_qubit_ratio": frame["initial_two_qubit_ratio"],
        }
    )


def final_metric_frame(frame):
    final_total = frame["final_total_gates"].replace(0, np.nan)
    final_qubits = frame["final_num_qubits"].replace(0, np.nan)
    return pd.DataFrame(
        {
            "num_qubits": frame["final_num_qubits"],
            "depth": frame["final_depth"],
            "two_qubit_gates": frame["final_two_qubit_gates"],
            "total_gates": frame["final_total_gates"],
            "two_qubit_depth": frame["final_two_qubit_depth"],
            "gate_density": frame["final_total_gates"] / final_qubits,
            "two_qubit_ratio": frame["final_two_qubit_gates"] / final_total,
        }
    ).fillna(0.0)


train_single_df = train_df[~train_df["chain_name"].str.contains("__", regex=False)]
scale_source = pd.concat(
    [initial_metric_frame(train_df), final_metric_frame(train_single_df)],
    ignore_index=True,
)
metric_scale = scale_source.quantile(0.95).replace(0, 1.0).to_dict()
metric_scale


## Load Saved Models

In [ ]:
def model_weight_from_path(path):
    match = re.search(r"optimizer_chain_ppo_([0-9.]+)\.zip$", path.name)
    if not match:
        return None
    return float(match.group(1))


model_paths = sorted(
    CHAIN_OPTIMIZER_DIR.glob("optimizer_chain_ppo_*.zip"),
    key=lambda path: model_weight_from_path(path),
)

model_info = pd.DataFrame(
    {
        "runtime_weight": [model_weight_from_path(path) for path in model_paths],
        "model_path": [path.name for path in model_paths],
    }
)

if model_info.empty:
    raise FileNotFoundError("No optimizer_chain_ppo_*.zip models found.")

model_info


## Evaluation Helpers

In [ ]:
def evaluate_policy(model, frame, method):
    env = OptimizerChainEnv(
        frame,
        OPTIMIZER_NAMES,
        CATEGORY_NAMES,
        metric_scale,
        shuffle_each_epoch=False,
    )
    selected = []
    invalid_actions = 0

    for circuit in sorted(frame["original_circuit"].unique()):
        obs, _ = env.reset(options={"circuit": circuit})
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))

        if info.get("invalid_action", False):
            invalid_actions += 1
            continue

        if not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))

        selected_chain = info["selected_chain"]
        row = frame[
            (frame["original_circuit"] == circuit)
            & (frame["chain_name"] == selected_chain)
        ].iloc[0].copy()
        row["method"] = method
        row["selected_chain"] = selected_chain
        selected.append(row)

    result = pd.DataFrame(selected)
    result.attrs["invalid_action_rate"] = invalid_actions / max(frame["original_circuit"].nunique(), 1)
    return result


def fixed_chain_rows(frame, chain_name, method):
    rows = frame[frame["chain_name"] == chain_name].copy()
    rows["method"] = method
    rows["selected_chain"] = chain_name
    return rows


def oracle_rows(frame, score_col):
    return frame.loc[frame.groupby("original_circuit")[score_col].idxmax()].copy()


def best_single_rows(frame, score_col="utility_reward"):
    single_rows = frame[~frame["chain_name"].str.contains("__", regex=False)].copy()
    return single_rows.loc[single_rows.groupby("original_circuit")[score_col].idxmax()].copy()


def summarize_selection(selected, method, split, runtime_weight=None):
    return {
        "method": method,
        "split": split,
        "runtime_weight": runtime_weight,
        "circuits": selected["original_circuit"].nunique(),
        "avg_reduction_pct": selected["two_qubit_gate_reduction_pct"].mean(),
        "avg_raw_reduction": selected["two_qubit_gate_reduction"].mean(),
        "avg_runtime_seconds": selected["total_runtime_seconds"].mean(),
        "avg_runtime_score": selected["runtime_score"].mean(),
        "avg_utility_at_0.00": utility_for_weight(selected, 0.00).mean(),
        "avg_utility_at_0.05": utility_for_weight(selected, 0.05).mean(),
        "avg_utility_at_0.10": utility_for_weight(selected, 0.10).mean(),
        "avg_utility_at_0.20": utility_for_weight(selected, 0.20).mean(),
        "avg_utility_at_0.35": utility_for_weight(selected, 0.35).mean(),
        "selected_unique_chains": selected["selected_chain"].nunique(),
    }


## Evaluate Models on Validation and Test

In [ ]:
all_selected = []
summary_rows = []

for _, info in model_info.iterrows():
    runtime_weight = float(info["runtime_weight"])
    model_path = CHAIN_OPTIMIZER_DIR / info["model_path"]
    model_label = f"PPO rw={runtime_weight:g}"

    env_for_load = OptimizerChainEnv(
        val_df,
        OPTIMIZER_NAMES,
        CATEGORY_NAMES,
        metric_scale,
        shuffle_each_epoch=False,
    )
    model = PPO.load(model_path, env=env_for_load)

    for split, frame in [("validation", val_df), ("test", test_df)]:
        selected = evaluate_policy(model, frame, method=model_label)
        selected["runtime_weight"] = runtime_weight
        selected["split"] = split
        all_selected.append(selected)
        summary_rows.append(
            summarize_selection(
                selected,
                method=model_label,
                split=split,
                runtime_weight=runtime_weight,
            )
        )

ppo_selected_all = pd.concat(all_selected, ignore_index=True)
ppo_summary = pd.DataFrame(summary_rows)
ppo_summary.sort_values(["split", "runtime_weight"])


## Baselines

Baselines are evaluated on the same validation/test circuits. The utility oracle here is based on the default utility weight `0.20`; reduction oracle is the best reduction regardless of runtime.

In [ ]:
baseline_rows = []
baseline_selected = []

for split, frame in [("validation", val_df), ("test", test_df)]:
    best_overall_chain = (
        train_df.groupby("chain_name")["utility_reward"]
        .mean()
        .sort_values(ascending=False)
        .index[0]
    )

    baseline_sets = [
        fixed_chain_rows(frame, "tket__wisq_bqskit", "tket__wisq_bqskit"),
        fixed_chain_rows(frame, "wisq_bqskit", "wisq_bqskit"),
        fixed_chain_rows(frame, best_overall_chain, "best overall train chain"),
        best_single_rows(frame, "utility_reward").assign(
            method="best single per circuit",
            selected_chain=lambda x: x["chain_name"],
        ),
        oracle_rows(frame, "utility_reward").assign(
            method="utility oracle rw=0.2",
            selected_chain=lambda x: x["chain_name"],
        ),
        oracle_rows(frame, "two_qubit_gate_reduction_pct").assign(
            method="reduction oracle",
            selected_chain=lambda x: x["chain_name"],
        ),
    ]

    for selected in baseline_sets:
        selected = selected.copy()
        selected["split"] = split
        selected["runtime_weight"] = np.nan
        baseline_selected.append(selected)
        baseline_rows.append(
            summarize_selection(
                selected,
                method=selected["method"].iloc[0],
                split=split,
                runtime_weight=np.nan,
            )
        )

baseline_selected_all = pd.concat(baseline_selected, ignore_index=True)
baseline_summary = pd.DataFrame(baseline_rows)

comparison_summary = pd.concat([ppo_summary, baseline_summary], ignore_index=True)
comparison_summary.sort_values(["split", "avg_reduction_pct", "avg_runtime_seconds"], ascending=[True, False, True])


## Validation Tradeoff Plot

Use validation to choose the reward weight. The test set should be used after you pick the model you want.

In [ ]:
val_plot = comparison_summary[comparison_summary["split"] == "validation"].copy()

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=val_plot,
    x="avg_runtime_seconds",
    y="avg_reduction_pct",
    hue="method",
    style=val_plot["method"].str.startswith("PPO"),
    s=120,
)

for _, row in val_plot.iterrows():
    label = f"{row['runtime_weight']:g}" if pd.notna(row["runtime_weight"]) else row["method"]
    plt.text(row["avg_runtime_seconds"], row["avg_reduction_pct"], " " + label, fontsize=9)

plt.xlabel("Average runtime seconds")
plt.ylabel("Average two-qubit gate reduction %")
plt.title("Validation Runtime vs Reduction Tradeoff")
plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Test Tradeoff Plot

In [ ]:
test_plot = comparison_summary[comparison_summary["split"] == "test"].copy()

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=test_plot,
    x="avg_runtime_seconds",
    y="avg_reduction_pct",
    hue="method",
    style=test_plot["method"].str.startswith("PPO"),
    s=120,
)

for _, row in test_plot.iterrows():
    label = f"{row['runtime_weight']:g}" if pd.notna(row["runtime_weight"]) else row["method"]
    plt.text(row["avg_runtime_seconds"], row["avg_reduction_pct"], " " + label, fontsize=9)

plt.xlabel("Average runtime seconds")
plt.ylabel("Average two-qubit gate reduction %")
plt.title("Test Runtime vs Reduction Tradeoff")
plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Utility Across Reward Weights

In [ ]:
utility_cols = [
    "avg_utility_at_0.00",
    "avg_utility_at_0.05",
    "avg_utility_at_0.10",
    "avg_utility_at_0.20",
    "avg_utility_at_0.35",
]

utility_view = comparison_summary[
    ["split", "method", "runtime_weight", *utility_cols]
].sort_values(["split", "method"])

utility_view


In [ ]:
ppo_test_utility = ppo_summary[ppo_summary["split"] == "test"].copy()
ppo_test_utility = ppo_test_utility.sort_values("runtime_weight")

plt.figure(figsize=(10, 5))
for col in utility_cols:
    weight_label = col.replace("avg_utility_at_", "utility at rw=")
    plt.plot(
        ppo_test_utility["runtime_weight"],
        ppo_test_utility[col],
        marker="o",
        label=weight_label,
    )

plt.xlabel("Runtime weight used to train model")
plt.ylabel("Average utility")
plt.title("Test Utility Recomputed Under Multiple Reward Weights")
plt.legend()
plt.tight_layout()
plt.show()


## Selected Chain Grid

In [ ]:
chain_grid_source = pd.concat([ppo_selected_all, baseline_selected_all], ignore_index=True)
chain_grid_source = chain_grid_source[chain_grid_source["split"] == "test"].copy()

chain_grid = chain_grid_source.pivot_table(
    index="method",
    columns="selected_chain",
    values="original_circuit",
    aggfunc="nunique",
    fill_value=0,
)
chain_grid = chain_grid.loc[:, chain_grid.sum(axis=0).sort_values(ascending=False).index]

display(chain_grid)

plt.figure(figsize=(max(12, 0.6 * chain_grid.shape[1]), max(5, 0.35 * chain_grid.shape[0])))
sns.heatmap(
    chain_grid,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Number of test circuits"},
)
plt.xlabel("Selected optimizer chain")
plt.ylabel("Method")
plt.title("Selected Chain Counts on Test Set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Per-Circuit Model Outputs

This table is useful when you want to inspect which circuits drive the tradeoff.

In [ ]:
per_circuit = ppo_selected_all[
    [
        "split",
        "runtime_weight",
        "original_circuit",
        "original_category",
        "selected_chain",
        "utility_reward",
        "two_qubit_gate_reduction_pct",
        "total_runtime_seconds",
    ]
].sort_values(["split", "original_circuit", "runtime_weight"])

per_circuit.head(50)
